# IRIS 데이터셋과 k-NN 분류기 실습

대표 입문용 다중 분류 데이터인 **IRIS** 로 **k-Nearest Neighbors (k-NN)** 분류기의 동작과 결정 경계를 살펴봅니다.

## 학습 목표
- `sklearn.datasets.load_iris` 로 표준 데이터셋 로드 및 구조 확인
- `KNeighborsClassifier` 로 k-NN 모델 학습
- `weights='uniform'` vs `'distance'` 의 결정 경계 차이 비교
- 메쉬그리드(`np.meshgrid`) + `contourf` 로 결정 영역 시각화

## 사용 라이브러리
`numpy`, `matplotlib`, `scikit-learn`

## 1. 데이터 로드

scikit-learn 에 내장된 IRIS 데이터셋(150 샘플 × 4 특성, 3 클래스) 을 가져옵니다.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn import neighbors, datasets

# IRIS 데이터셋 가져오기
iris = datasets.load_iris()

print(iris)
print(iris.data.shape)

## 2. 시각화용 특성 선택

전체 4개 특성 중 **꽃받침 길이/너비(첫 두 컬럼)** 만 골라 2D 평면에서 결정 경계를 그릴 수 있도록 준비합니다. 라벨 `y` 는 0/1/2 (setosa/versicolor/virginica) 입니다.

In [ ]:

# 데이터 시각화를 위해 특성 1, 2 (Sepal 길이 너비)만을 사용함
X = iris.data[:, :2]
y = iris.target


## 3. k-NN 결정 경계 시각화

`weights` 옵션에 따라 같은 k 라도 결정 경계가 달라집니다. 메쉬그리드의 모든 좌표에 `predict()` 를 호출해 영역을 색칠합니다.

| `weights` | 동작 |
|---|---|
| `'uniform'` | 가까운 k 개 이웃에 동일한 표(vote) 부여 |
| `'distance'` | 거리에 반비례하는 가중치 — 가까운 이웃의 영향력이 더 큼 |

In [ ]:
# 예상되는 데이터 영역을 h 크기의 메쉬형태로 구분
h = .02


n_neighbors = 5

# 결과를 비교하기 위해 두개 그림을 출력
fig, axs = plt.subplots(1,2, figsize=(16,5))

for idx, weights in enumerate(['uniform', 'distance']):
    # scitkit-learn에서 k-NN 모델을 가져옴
    clf = neighbors.KNeighborsClassifier(n_neighbors, weights=weights)
    clf.fit(X, y)

    # 데이터의 최대, 최소 영역을 설정
    x_min = X[:, 0].min() - 1
    x_max = X[:, 0].max() + 1

    y_min = X[:, 1].min() - 1
    y_max = X[:, 1].max() + 1

    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

    # xx, yy array 크기 확인
    print(xx.shape)

    # xx.ravel := xx.reshape(-1) --> 1차원 벡터로 평탄화
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])

    # Z를 다시 컨투어 가시과를 위해 메쉬 데이터로 복원
    Z_out = Z.reshape(xx.shape)

    # fig[idx]= plt.figure(figsize=(8, 6))
    axs[idx].contourf(xx, yy, Z_out, cmap='cool')

    # Plot also the training points
    scatter = axs[idx].scatter(x=X[:, 0], y=X[:, 1], cmap='cool', c=y,
                     label=iris.target_names[y], alpha=1.0, edgecolor="black")
    axs[idx].set_xlim(xx.min(), xx.max())
    axs[idx].set_ylim(yy.min(), yy.max())
    axs[idx].set_title("3-Class classification (k = %i, weights = '%s')"
              % (n_neighbors, weights))
    axs[idx].set_xlabel(iris.feature_names[0])
    axs[idx].set_ylabel(iris.feature_names[1])

    # labels index --> 아이리스 이름으로 변환
    handles, labels = scatter.legend_elements(prop='colors')
    for i in [0,1,2]:
       labels[i] = iris.target_names[i]

    axs[idx].legend(handles, labels)
    #axs[idx].add_artist(legend)


plt.show()